# E2: LViT-TW No-Text on Preprocessed BTXRD 224x224

Clean Kaggle runner for the image-only LViT baseline. This notebook trains tumor-only and evaluates on the full validation/test splits for Q3 normal false-positive metrics.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/lehngoc/BTXRD-LViT.git"
BRANCH = "model/e2e3-lvit-224"
REPO_ROOT = Path("/kaggle/working/BTXRD-LViT")
CONFIG = REPO_ROOT / "configs/train_lvit_tw_preprocessed_224.yaml"
OUTPUT_DIR = Path("/kaggle/working/experiments/E2_lvit_tw_preprocessed_224_tumor_only")
SMOKE_OUTPUT_DIR = Path("/kaggle/working/experiments/_smoke_E2_lvit_tw")
DATA_ROOT = Path("/kaggle/input/datasets/lehngoc/btxrd-preprocessed-dataset/btxrd-preprocessed")

In [ ]:
!rm -rf {REPO_ROOT}
!git clone -b {BRANCH} {REPO_URL} {REPO_ROOT}
%cd {REPO_ROOT}

In [ ]:
import copy
import yaml

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"DATA_ROOT does not exist: {DATA_ROOT}. Attach the BTXRD preprocessed Kaggle dataset or edit DATA_ROOT.")

cfg = yaml.safe_load(CONFIG.read_text())
cfg["data"]["root_dir"] = str(DATA_ROOT)
cfg["training"]["device"] = "cuda"
cfg["training"]["num_workers"] = 2
cfg["training"]["output_dir"] = str(OUTPUT_DIR)
runtime_config = Path("/kaggle/working/e2_lvit_tw_runtime.yaml")
runtime_config.write_text(yaml.safe_dump(cfg, sort_keys=False))

smoke_cfg = copy.deepcopy(cfg)
smoke_cfg["training"]["output_dir"] = str(SMOKE_OUTPUT_DIR)
smoke_config = Path("/kaggle/working/e2_lvit_tw_smoke.yaml")
smoke_config.write_text(yaml.safe_dump(smoke_cfg, sort_keys=False))

runtime_config, smoke_config

In [ ]:
# Smoke test: one epoch with tiny samples, written to a separate smoke directory.
!rm -rf {SMOKE_OUTPUT_DIR}
!python -m src.training.train_lvit_tw --config {smoke_config} --epochs 1 --max-train-samples 4 --max-val-samples 4 --max-test-samples 4 --device cuda

In [ ]:
# Full E2 run, written to a clean output directory.
!rm -rf {OUTPUT_DIR}
!python -m src.training.train_lvit_tw --config {runtime_config} --device cuda

In [ ]:
zip_path = Path("/kaggle/working/E2_lvit_tw_preprocessed_224_metrics_only.zip")
!cd {OUTPUT_DIR} && zip -r {zip_path} history.csv best_summary.json val_metrics.json test_metrics.json test_threshold_sweep_metrics.json test_metrics_thr*.json config.json
zip_path